In [ ]:
%pip install "monai>=1.3.0,<2" "gcsfs>=2023.1.0,<2027"

In [ ]:
from google.colab import auth
auth.authenticate_user()
!gcloud config set project clinimcl
!nvidia-smi

import torch, monai
print(f"[env] {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu'} | torch {torch.__version__} | monai {monai.__version__}")

In [ ]:
import os
if not os.path.exists("model.py"):
    !git clone https://github.com/cadenroberts/ClinImCL.git /content/ClinImCL
    %cd /content/ClinImCL

In [ ]:
import os, io, re, time, math, random, subprocess, tempfile
from collections import defaultdict
from functools import lru_cache
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.amp import GradScaler, autocast
from torch.serialization import add_safe_globals
import gcsfs

from model import ClinImCL, IMG

try:
    from monai.data.meta_tensor import MetaTensor
    from monai.utils.enums import TraceKeys
    add_safe_globals([MetaTensor, TraceKeys, np.ndarray])
except ImportError:
    add_safe_globals([np.ndarray])

# ── Config ──────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

device     = "cuda" if torch.cuda.is_available() else "cpu"
BATCH      = 8
EPOCHS     = 20
LR         = 3e-4
TEMP       = 0.07
WARMUP_EP  = 3
FRAC       = 0.25
LOG_EVERY  = 50
GCS_DATA   = "gs://clinimcl-data/OASIS3/preprocessed/"
GCS_CKPT   = "gs://clinimcl-data/checkpoints/"

torch.backends.cudnn.benchmark = True
torch.set_float32_matmul_precision("high")
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

# ── GCS index ──────────────────────────────────────────────────────
fs = gcsfs.GCSFileSystem(token="google_default")
pt_files = [p if p.startswith("gs://") else f"gs://{p}"
            for p in fs.ls(GCS_DATA) if p.lower().endswith(".pt")]

_re = re.compile(r"(OAS3\d+)\D*?_d(\d+)\.pt$", re.IGNORECASE)
subjects = defaultdict(list)
for p in pt_files:
    m = _re.search(os.path.basename(p))
    if m:
        subjects[m.group(1)].append((int(m.group(2)), p))
for v in subjects.values():
    v.sort()
pairs = {s: tp for s, tp in subjects.items() if len(tp) >= 2}
pair_ids = list(pairs)
print(f"[data] {len(pt_files)} files | {len(subjects)} subjects | {len(pair_ids)} with >=2 tps")
if not pair_ids:
    raise RuntimeError("No subjects with >=2 timepoints found — cannot form contrastive pairs")

# ── Helpers ─────────────────────────────────────────────────────────
def _gcs_read(url, retries=3):
    for attempt in range(retries):
        try:
            with fs.open(url, "rb") as f:
                return f.read()
        except Exception:
            if attempt == retries - 1:
                raise
            time.sleep(2 ** attempt)

# ~3.5 MiB per cached 96^3 float32 volume; 512 entries ≈ 1.75 GB RAM
@lru_cache(maxsize=512)
def load_vol(url):
    vol = torch.load(io.BytesIO(_gcs_read(url)), map_location="cpu", weights_only=True)
    vol = torch.as_tensor(vol, dtype=torch.float32)
    if vol.ndim == 3: vol = vol.unsqueeze(0)
    if vol.shape[0] != 1: vol = vol[:1]
    if vol.shape[1:] != (IMG, IMG, IMG):
        vol = F.interpolate(vol.unsqueeze(0), size=(IMG,IMG,IMG),
                            mode="trilinear", align_corners=False).squeeze(0)
    return vol.contiguous()

def augment(x):
    x = x.clone()
    if random.random() < 0.5:
        x = torch.flip(x, dims=[random.choice([1, 2, 3])])
    if random.random() < 0.3:
        x = x * random.uniform(0.9, 1.1) + random.uniform(-0.1, 0.1)
    if random.random() < 0.3:
        x = x + torch.randn_like(x) * random.uniform(0.01, 0.05)
    return x.clamp(0.0, 1.0)

# ── Dataset ─────────────────────────────────────────────────────────
class PairDataset(Dataset):
    def __init__(self, pairs_dict, items):
        self.p, self.items = pairs_dict, items
    def __len__(self):
        return len(self.items)
    def __getitem__(self, idx):
        (_, a), (_, b) = random.sample(self.p[self.items[idx]], 2)
        return augment(load_vol(a)), augment(load_vol(b))

def epoch_loader(ep):
    rng = random.Random(ep + 12345)
    ids = rng.sample(pair_ids, max(1, int(FRAC * len(pair_ids))))
    batched_ids = []
    for _ in range(4):
        perm = ids.copy()
        rng.shuffle(perm)
        batched_ids.extend(perm)
    dl = DataLoader(PairDataset(pairs, batched_ids), batch_size=BATCH,
                    shuffle=False, num_workers=0,
                    pin_memory=(device=="cuda"), drop_last=True)
    return dl

model = ClinImCL().to(device)
print(f"[model] {sum(p.numel() for p in model.parameters())/1e6:.2f}M params")

def info_nce(z1, z2):
    logits = (z1 @ z2.t()) / TEMP
    tgt = torch.arange(z1.size(0), device=z1.device)
    return 0.5 * (F.cross_entropy(logits, tgt) + F.cross_entropy(logits.t(), tgt))

# ── Optimizer + schedule ───────────────────────────────────────────
opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scaler = GradScaler(device, enabled=(device=="cuda"))
n_sampled = max(1, int(FRAC * len(pair_ids)))
steps_ep = (n_sampled * 4) // BATCH
wu = WARMUP_EP * steps_ep
total = EPOCHS * steps_ep

def lr_fn(s):
    if s < wu: return s / max(1, wu)
    return 0.5 * (1 + math.cos(math.pi * (s - wu) / max(1, total - wu)))
sched = torch.optim.lr_scheduler.LambdaLR(opt, lr_fn)

# ── Resume from latest checkpoint (if any) ─────────────────────────
start_ep = 1
gs = 0
_ep_re = re.compile(r"_ep(\d+)")
def _ckpt_epoch(path):
    m = _ep_re.search(path)
    return int(m.group(1)) if m else -1
ckpt_files = sorted([p for p in fs.ls(GCS_CKPT) if p.endswith(".pth")], key=_ckpt_epoch)
if ckpt_files:
    latest = ckpt_files[-1]
    if not latest.startswith("gs://"):
        latest = f"gs://{latest}"
    print(f"[resume] loading {latest}")
    with fs.open(latest, "rb") as f:
        ckpt = torch.load(io.BytesIO(f.read()), map_location=device, weights_only=True)
    _cur_cfg = dict(IMG=IMG, TEMP=TEMP, proj=128, base=32, FRAC=FRAC, BATCH=BATCH)
    _saved_cfg = ckpt.get("cfg", {})
    for _k in _cur_cfg:
        if _k in _saved_cfg and _saved_cfg[_k] != _cur_cfg[_k]:
            print(f"[resume] WARNING cfg mismatch: {_k}={_saved_cfg[_k]} (ckpt) vs {_cur_cfg[_k]} (current)")
    model.load_state_dict(ckpt["model"])
    opt.load_state_dict(ckpt["optimizer"])
    sched.load_state_dict(ckpt["scheduler"])
    if "scaler" in ckpt:
        scaler.load_state_dict(ckpt["scaler"])
    start_ep = ckpt["epoch"] + 1
    gs = ckpt.get("gs", (start_ep - 1) * steps_ep)
    print(f"[resume] resuming from epoch {start_ep}, gs={gs}")

# ── Train ──────────────────────────────────────────────────────────
_save_dir = "/content" if os.path.isdir("/content") else tempfile.gettempdir()
t0_all = time.time()
for ep in range(start_ep, EPOCHS + 1):
    dl = epoch_loader(ep)
    model.train()
    losses, t0, step = [], time.time(), 0

    for x1, x2 in dl:
        x1, x2 = x1.to(device, non_blocking=True), x2.to(device, non_blocking=True)
        opt.zero_grad(set_to_none=True)
        with autocast(device_type=device, enabled=(device=="cuda")):
            loss = info_nce(model(x1)[0], model(x2)[0])
        scaler.scale(loss).backward()
        scaler.step(opt); scaler.update(); sched.step()
        losses.append(loss.item()); gs += 1; step += 1
        if LOG_EVERY and gs % LOG_EVERY == 0:
            print(f"  [ep {ep:02d} step {gs:05d}] loss={np.mean(losses[-50:]):.4f} lr={sched.get_last_lr()[0]:.2e}")

    if not losses:
        print(f"[ep {ep:02d}] {time.time()-t0:.0f}s | steps=0 | no batches (dataset too small for batch size)")
        continue

    print(f"[ep {ep:02d}] {time.time()-t0:.0f}s | steps={step} | loss={np.mean(losses):.4f}")
    load_vol.cache_clear()

    path = os.path.join(_save_dir, f"clinimcl_ep{ep:02d}.pth")
    torch.save({"epoch": ep, "gs": gs, "model": model.state_dict(),
                "optimizer": opt.state_dict(), "scheduler": sched.state_dict(),
                "scaler": scaler.state_dict(),
                "cfg": dict(IMG=IMG, TEMP=TEMP, proj=128, base=32, FRAC=FRAC, BATCH=BATCH)}, path)
    dst = f"{GCS_CKPT}clinimcl_ep{ep:02d}_{time.strftime('%Y%m%d_%H%M%S')}.pth"
    ret = subprocess.run(["gsutil", "cp", path, dst])
    if ret.returncode == 0:
        os.remove(path)
        print(f"[upload] {dst}")
    else:
        print(f"[save] {path}")

print(f"\nDone in {time.time()-t0_all:.0f}s")